# 🎨 AbiArt — Kaggle GPU Backend (Automatic1111 SD WebUI API)

Ce notebook transforme un environnement **Kaggle GPU gratuit (30h/semaine)** en serveur backend haute performance pour votre studio **AbiArt**.

### ⚡ Instructions rapides :
1. Dans les paramètres à droite sur Kaggle : activez **Accelerator → GPU T4 x2** (ou P100) et **Internet → ON**.
2. Exécutez les cellules ci-dessous dans l'ordre (ou cliquez sur **Run All**).
3. À la dernière cellule, copiez l'URL publique générée (`https://xxxx.trycloudflare.com`).
4. Ouvrez AbiArt, sélectionnez le backend **Kaggle**, collez l'URL et cliquez sur **Connect** !

In [ ]:
# 1. Vérification du GPU
!nvidia-smi

In [ ]:
# 2. Installation de Stable Diffusion WebUI (Automatic1111) et dépendances
import os
%cd /kaggle/working

# Cloner SD WebUI si non présent
if not os.path.exists('/kaggle/working/stable-diffusion-webui'):
    !git clone https://github.com/AUTOMATIC1111/stable-diffusion-webui.git /kaggle/working/stable-diffusion-webui

%cd /kaggle/working/stable-diffusion-webui

# Installer extensions populaires (ADetailer pour visages/mains, ControlNet)
!git clone --depth 1 https://github.com/Bing-su/adetailer.git extensions/adetailer 2>/dev/null || true
!git clone --depth 1 https://github.com/Mikubill/sd-webui-controlnet.git extensions/sd-webui-controlnet 2>/dev/null || true

# Dépendances Python requises
!pip install -q xformers==0.0.23.post1 ultralytics mediapipe pyngrok

print("✓ Automatic1111 et extensions prêts !")

In [ ]:
# 3. Téléchargement des Modèles de départ (Checkpoints & LoRAs)
%cd /kaggle/working/stable-diffusion-webui/models/Stable-diffusion

# Téléchargement d'un modèle SDXL anime haute qualité (Animagine XL 3.1)
if not os.path.exists('animagine-xl-3.1.safetensors'):
    print("Téléchargement d'Animagine XL 3.1...")
    !wget -q -c https://huggingface.co/cagliostrolab/animagine-xl-3.1/resolve/main/animagine-xl-3.1.safetensors -O animagine-xl-3.1.safetensors

# Modèle VAE SDXL
%cd /kaggle/working/stable-diffusion-webui/models/VAE
if not os.path.exists('sdxl_vae.safetensors'):
    !wget -q -c https://huggingface.co/stabilityai/sdxl-vae/resolve/main/sdxl_vae.safetensors -O sdxl_vae.safetensors

%cd /kaggle/working/stable-diffusion-webui
print("✓ Modèles installés dans models/Stable-diffusion !")

In [ ]:
# 4. Installation de Cloudflare Tunnel (100% gratuit, sans compte)
!wget -q -c https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb >/dev/null 2>&1
print("✓ Cloudflared installé !")

In [ ]:
# 5. Démarrage du Tunnel et du Serveur Automatic1111 avec API
import subprocess
import time
import re
import threading

# Démarrer le tunnel Cloudflare en arrière-plan
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:7860"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

def print_tunnel_url():
    for line in iter(tunnel_proc.stderr.readline, ''):
        match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
        if match:
            url = match.group(0)
            print("\n" + "="*60)
            print("🎨 URL KAGGLE POUR ABIART :")
            print(f"   >>>  {url}  <<<")
            print("="*60)
            print("Copiez cette URL et collez-la dans AbiArt → Backend: Kaggle → Connect\n")
            break

threading.Thread(target=print_tunnel_url, daemon=True).start()
time.sleep(2)

# Lancement de l'API Automatic1111
!python launch.py \
    --enable-insecure-extension-access \
    --skip-torch-cuda-test \
    --no-half-vae \
    --xformers \
    --api \
    --listen \
    --port 7860